In [1]:
!pip install tweepy

In [2]:
!git --version

git version 2.56.0.windows.4


In [3]:
!pip install git+https://github.com/tweepy/tweepy.git

  Cloning https://github.com/tweepy/tweepy.git to c:\users\hp\appdata\local\temp\pip-req-build-yld9jp_q
  Resolved https://github.com/tweepy/tweepy.git to commit c1978d643ecce491929084e4290b35f57e4921ad
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/tweepy/tweepy.git 'C:\Users\HP\AppData\Local\Temp\pip-req-build-yld9jp_q'


In [ ]:
import tweepy
import pandas as pd
import os
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timedelta

#consumer_key = "************" # API/Consumer key 
#consumer_secret = "*********" # API/Consumer Secret Key
#access_token = "***********"    # Access token key
#access_token_secret = "*************" # APIAccess token Secret key

#Pass in our twitter API authentication key
auth = tweepy.OAuth1UserHandler(
    consumer_key, consumer_secret,
    access_token, access_token_secret
)

client = tweepy.Client(
    bearer_token=BEARER_TOKEN,
    wait_on_rate_limit=True
)
#Instantiate the tweepy API
api = tweepy.API(auth, wait_on_rate_limit=True)

# Search Query Construction ---
# Technology keyword groups: AI, ML, CC, Hadoop, Python (OR within group, OR across groups)
TECH_TERMS = (
    '("artificial intelligence" OR "AI" OR "intelligent systems" OR '
    '"machine learning" OR "ML" OR "deep learning" OR "neural networks" OR '
    '"cloud computing" OR "cloud services" OR "AWS" OR "Azure" OR "cloud infrastructure" OR '
    '"Hadoop" OR "HDFS" OR "MapReduce" OR "big data Hadoop" OR '
    '"Python programming" OR "Python coding" OR "Python" OR "Python developer")'
)

# Exclusion filters drop replies and link-only tweets;
# retweets retained and flagged later (for volume analysis)
EXCLUSIONS = '-is:reply -has:links'

# Geographic filter: India bounding box [west_lon south_lat east_lon north_lat]
GEO_BBOX = 'bounding_box:[68.1 8.1 97.4 37.1]'

# Language filter : English only
LANG = 'lang:en'

# Final query string
SEARCH_QUERY = f'{TECH_TERMS} {EXCLUSIONS} {GEO_BBOX} {LANG}'

START_TIME  = '2026-01-11T00:00:00Z'
END_TIME    = '2026-02-10T23:59:59Z'
MAX_TWEETS  = 1_600_000          
BATCH_SIZE  = 500                

# Attribute Extraction ---
TWEET_FIELDS = ['created_at', 'lang', 'public_metrics', 'geo',
                'place', 'entities', 'author_id']
USER_FIELDS  = ['name', 'username', 'description', 'public_metrics',
                'verified', 'created_at', 'location']
PLACE_FIELDS = ['full_name', 'country', 'geo', 'place_type', 'contained_within']

def extract_attributes(tweet, user, place):
    coords = (place.geo.get('coordinates', [None, None])
              if place and place.geo else [None, None])
    return {
        'Tweet ID':          tweet.id,
        'Date Created':      tweet.created_at,
        'User ID':           user.id if user else None,
        'Name':              user.name if user else None,
        'Username':          user.username if user else None,
        'Tweet':             tweet.text,
        'Language':          tweet.lang,
        'Retweets':          tweet.public_metrics['retweet_count'],
        'Likes':             tweet.public_metrics['like_count'],
        'Followers':         user.public_metrics['followers_count'] if user else None,
        'Verified':          user.verified if user else None,
        'Account Created':   user.created_at if user else None,
        'Bio':               user.description if user else None,
        'City':              place.full_name if place else None,
        'Country':           place.country if place else None,
        'Latitude':          coords[1],
        'Longitude':         coords[0],
        'Type':              'retweet' if tweet.text.startswith('RT @') else 'original',
        'Tweet URL':         f'https://twitter.com/i/web/status/{tweet.id}',
    }

# Paginated Collection ---
records, next_token = [], None

while len(records) < MAX_TWEETS:
    response = client.search_all_tweets(
        query=SEARCH_QUERY,
        start_time=START_TIME,
        end_time=END_TIME,
        max_results=BATCH_SIZE,
        tweet_fields=TWEET_FIELDS,
        user_fields=USER_FIELDS,
        place_fields=PLACE_FIELDS,
        expansions=['author_id', 'geo.place_id'],
        next_token=next_token,
    )

    if not response.data:
        break

    users  = {u.id: u for u in response.includes.get('users', [])}
    places = {p.id: p for p in response.includes.get('places', [])}

    for tw in response.data:
        user  = users.get(tw.author_id)
        place = places.get(tw.geo.place_id) if tw.geo else None
        records.append(extract_attributes(tw, user, place))

    next_token = response.meta.get('next_token')
    print(f"Collected {len(records):,} tweets | next_token={next_token}")
    if not next_token:
        break

# Build DataFrame and Deduplicate ---
df = pd.DataFrame(records)
print(f"Raw collected: {len(df):,} tweets")

# Exact duplicate removal by Tweet ID
df = df.drop_duplicates(subset=['Tweet ID'], keep='first')
print(f"After exact deduplication: {len(df):,} tweets")

# Near-duplicate flag (MinHash LSH applied downstream in Spark)
df['is_near_duplicate'] = False

# Persist Raw Collection to HDFS ---
out_path = 'hdfs:///data/raw/year=2026/month=01/day=*/hour=*/'
df.to_parquet(f'raw_tweets_{datetime.utcnow():%Y%m%d}.parquet', index=False)
print(f"Saved to {out_path}")

hdfs_path = 'hdfs:///data/raw/year=2026/month=01/'
table = pa.Table.from_pandas(df, preserve_index=False)
pq.write_to_dataset(
    table,
    root_path=hdfs_path,
    partition_cols=['Date Created'],
    existing_data_behavior='overwrite_or_ignore',
    compression='snappy',
)
print(f"Saved {len(df):,} tweets to {hdfs_path}")

In [ ]:
class FlumeFeed(tweepy.StreamingClient):
    def on_tweet(self, tweet):
        # Mirror extract_attributes() schema; push to Flume HTTP source
        pass

flume_stream = FlumeFeed(BEARER_TOKEN)
flume_stream.add_rules(tweepy.StreamRule(SEARCH_QUERY))
flume_stream.filter(tweet_fields=TWEET_FIELDS,
                    expansions=['author_id', 'geo.place_id'])

In [ ]:
from google.colab import files
tweets_df.to_csv('tweets.csv') 
files.download('tweets.csv')